# Step 20.2 — Save optimizer state：从状态树到文件，再按路径恢复

在仓库根目录同步母版：

```bash
git pull
```

再建立独立工作副本，不覆盖已有笔记：

```bash
cp -n notebooks/88_step20_2_save_optimizer_state.ipynb notebooks/work/88_step20_2_save_optimizer_state_$(date +%Y%m%d_%H%M%S)_work.ipynb
```

以上是未来命令参考，本次未执行。环境沿用项目Python 3.12 / `.venv`、固定 **MLX v0.32.2**。所有代码单元未执行、输出为空；没有创建模型、读取语料或保存／加载任何真实状态文件。

2026-11-17 教学修订：**20.2累计2/2＝历史状态校验修正1次＋本次教学整理1次**，不重置历史。保留原format_version=2、三个保存／恢复函数和20.3/20.4衔接；旧模型移到可选附录，只改本节。


## 阅读路线与文件职责

[20.1](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/3255a2975b7e716d95c4f92d76f25e21256e7f66/notebooks/87_step20_1_save_model_weights.ipynb)保存模型权重；本节保存Adam用于下一次更新的状态。先看一棵微型树怎样对应两份文件，再读flatten、save、restore三段代码。完整MiniGPT只是已学前置，放在附录，不必先重读数百行模型代码才能理解序列化。

| 内容 | 本节负责什么？ |
|---|---|
| 正文 | 状态叶子的路径／类型、NPZ与manifest映射、目标匹配检查 |
| 可选附录 | 同一模型的完整定义、显式Adam配置与初始化模板 |
| 注释调用参考 | 说明未来需提供什么真实对象／路径；本轮不调用 |

本节不是完整checkpoint系统。文件不会自动附带模型权重、架构、完整词表、随机数或sampler状态；[20.3](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/3255a2975b7e716d95c4f92d76f25e21256e7f66/notebooks/89_step20_3_save_configuration_metadata.ipynb)另记来源与配对元数据，后续恢复流程还需这些信息。这里不改后续文件或训练协议。


## 1. 相同权重，不代表相同的下一次Adam更新

模型权重θ决定当前forward。Adam还记住梯度历史：一阶矩m和二阶矩v。逐元素的更新为：

$$
m_t=\beta_1m_{t-1}+(1-\beta_1)g_t,\qquad
v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2.
$$

g是本次梯度，m/v与对应参数同shape。启用bias correction时还用到更新计数t；本课显式bias_correction=True，固定版本默认False。学习率、eps等也参与更新，公式不是“只有权重就够了”。

只恢复θ，却把m/v与step重新置零，相当于从相同权重启动一个新的优化过程，通常不是接着原来的下一步。因此必须让权重与optimizer state来自同一完成更新边界；仅有相同shape不能证明它们是同一次训练的配对。


## 2. 零状态、训练后状态与“标量数组”

固定MLX v0.32.2的Optimizer构造时创建 `step = mx.array(0, mx.uint64)`。Adam把当前learning_rate也放进state为MLX数组；`optimizer.init(model.trainable_parameters())`再为各参数建立同shape的零m/v。这是初始化模板，不是训练结果。

| 叶子 | 刚构造／init后的含义 | 类型与shape |
|---|---|---|
| step | 已进行的更新数为0；之后每次update推进 | **零维MLX array，uint64，shape ()** |
| learning_rate | 当前学习率快照 | 本课标量学习率为零维浮点array，shape () |
| 参数路径下的m、v | 初始化时全零；训练后保存对应历史 | 与该参数相同shape／对应数组dtype |

“标量”描述只有一个数，不代表Python int/float。step的shape是()而不是(1,)；转为JSON shape元数据时，`list(())`得到[]。不要为方便讲解而先`.item()`把step改成Python整数再保存，这会改变原协议。

参数为(C_out,C_in)时m/v也为(C_out,C_in)；step虽不是可训练模型参数，却仍是**数组叶子**。本节NPZ保存所有数组，包含m/v、step和当前learning_rate。JSON里的python_scalars只装状态树中实际存在的Python标量；当前标准Adam通常没有这类叶子。


## 3. 只用一棵微型树看映射

为说明容器与文件关系，假设模型只有一个shape为(2,)的weight参数，状态树按下列顺序排列。数值是纸面示意，不来自任何真实优化器，也不替换本项目MiniGPT：

```text
state
├─ step             → array(7),           shape (),   uint64
├─ learning_rate    → array(0.0003),      shape (),   float32
└─ weight
   ├─ m             → array([0.1,-0.2]),  shape (2,), float32
   └─ v             → array([0.01,0.04]), shape (2,), float32
```

`tree_flatten`来自mlx.utils，返回“路径＋叶子”的列表。它展开dict/list/tuple容器，不把数组的数据展平。weight.m这个字符串记录树中weight→m的位置；真实模型还可能有 `blocks.0.attn.qkv_proj.weight.m` 这样的路径，其中0表示列表位置。

| 假设遍历序号 | 树路径path | NPZ键key | shape元数据 | dtype字符串 |
|---:|---|---|---|---|
| 0 | step | state_array_0000 | [] | mlx.core.uint64 |
| 1 | learning_rate | state_array_0001 | [] | mlx.core.float32 |
| 2 | weight.m | state_array_0002 | [2] | mlx.core.float32 |
| 3 | weight.v | state_array_0003 | [2] | mlx.core.float32 |

key只是本次保存分配的数组名字；真实state的遍历顺序变了，同一路径的序号也可能变，不能硬编码“0000永远是什么”。恢复时靠manifest中的path↔key关系，不依赖NPZ内部排列顺序。

树路径承担本课dict/list结构的描述；这不是任意Python对象序列化。tree_unflatten按路径重建嵌套结构、将数字索引解释为列表，本课使用的模块参数命名与此约定匹配，不额外保存类对象或通用容器schema。


### Flatten函数：先分数组与真正的Python标量

第一个分支检查isinstance(value,mx.array)，因此零维step也先归入array_items。第二个分支才接受int、float、bool、str或None，保存路径、值及python_type；其他叶子类型直接报错，不静默丢弃。

返回的array_items仍持有原数组引用，scalar_items是可写进JSON的Python记录。此函数本身不写文件、不执行update，也不把所有数组拼成一个长向量。下面只有导入与定义。


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json

import mlx.core as mx
from mlx.utils import tree_flatten, tree_unflatten

def flatten_optimizer_state(state):
    flattened = tree_flatten(state)
    array_items = []
    scalar_items = []
    for path, value in flattened:
        path_text = str(path)
        # In MLX v0.32.2 Adam state, step and current learning_rate are MLX arrays
        # (including the zero-dimensional uint64 step), so they belong in NPZ too.
        if isinstance(value, mx.array):
            array_items.append((path_text, value))
        elif isinstance(value, (int, float, bool, str)) or value is None:
            scalar_items.append({"path": path_text, "value": value, "python_type": type(value).__name__})
        else:
            raise TypeError(f"Unsupported optimizer-state leaf at {path!r}: {type(value)!r}")
    return array_items, scalar_items


## 4. 两个文件各自保存什么？

**Serialization（序列化）**是把内存对象转换为可保存表示。本协议版本仍为2：

| 文件 | 内容 | 不包含什么？ |
|---|---|---|
| optimizer_state.npz | key→MLX数组：m/v、step、当前learning_rate及其他实际数组叶子 | 不知道数组原来位于树的哪个路径 |
| optimizer_state.json | format_version、optimizer_class、array_file、arrays记录、python_scalars、note | 不存m/v或step的数组数值，不存完整优化器对象 |

微型树中step的JSON记录是下面这个**manifest中单条数组描述的片段**；数值7仍在NPZ数组里，不能在这里另存一个Python step：

```json
{"key": "state_array_0000", "path": "step", "shape": [], "dtype": "mlx.core.uint64"}
```

这个示意树的四个叶子都是数组，故python_scalars为空列表。若某个支持的状态扩展确实有Python标量，它才进python_scalars，并保留类型名；不能因为step读起来像整数，就认定它属于那一分支。

当前learning_rate快照不包含完整schedule规则或后续序列。betas、eps、bias_correction是Adam对象的其他配置，不由这些状态叶子完整表示，必须另行保留。


### 保存函数分四段读

1. 在output_root/optimizer_states下创建时间戳**新目录**。exist_ok=False拒绝复用同名目录；UTC时间戳本身仍可能碰撞。
2. `mx.eval(optimizer.state)`完成待保存数组，再flatten。求值不是训练，也不把零状态变成已训练状态。
3. enumerate给每个数组分配state_array_XXXX键，同时记录原path、list(shape)与str(dtype)；`mx.savez(..., **arrays)`把键和数组写进NPZ。
4. 构造manifest，以open("x")独占创建JSON，最后返回两条实际路径。

这里的output_root是**输出根目录**。若要与前后节统一放在results，传project_root/"results"；与20.1接受project_root后自己追加results的接口不同，不要多加或漏加一层。

新目录与x模式防止正常调用覆盖已有记录，但两份文件不是一个原子事务。写到一半失败可能留下不完整目录，不能自动把它当完整产物；本函数不清理、覆盖或重试旧目录。


In [ ]:
def save_optimizer_state_to_new_directory(optimizer, output_root):
    root = Path(output_root) / "optimizer_states"
    root.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    output_dir = root / f"adam_state_{stamp}"
    output_dir.mkdir(exist_ok=False)

    mx.eval(optimizer.state)
    array_items, scalar_items = flatten_optimizer_state(optimizer.state)
    arrays = {}
    array_manifest = []
    for index, (path, value) in enumerate(array_items):
        key = f"state_array_{index:04d}"
        arrays[key] = value
        array_manifest.append({"key": key, "path": path, "shape": list(value.shape), "dtype": str(value.dtype)})

    arrays_path = output_dir / "optimizer_state.npz"
    manifest_path = output_dir / "optimizer_state.json"
    # NPZ stores all MLX array leaves, including m/v, the scalar uint64 step,
    # and the current learning_rate snapshot when present in optimizer.state.
    mx.savez(str(arrays_path), **arrays)
    manifest = {
        "format_version": 2,
        "optimizer_class": type(optimizer).__name__,
        "array_file": arrays_path.name,
        "arrays": array_manifest,
        "python_scalars": scalar_items,
        "note": "Optimizer state only. Adam hyperparameters/scheduler, model weights, RNG, sampler and data state are separate.",
    }
    with manifest_path.open("x", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    return arrays_path, manifest_path


## 5. 恢复先检查两件不同的事

第一层：**文件与自己的manifest是否相符？** 加载JSON后检查format_version=2与optimizer_class，再从NPZ按key取得数组；每个数组的shape/dtype须与manifest一致。缺少key会报错，不凭空补零。Python标量也检查实际类型名，NoneType需要值确为None。

第二层：**这些内容能否装进当前已初始化的optimizer？** 当前模型结构与dtype给出目标模板。比较当前state与保存内容的路径集合；然后逐路径比较数组／非数组类别、数组shape和dtype，或非数组精确Python类型。这里的“数组身份”指类别，不要求保存前后是同一个Python对象。

两层不能合并：另一宽度模型的文件完全可能与自己的manifest自洽，但仍不匹配当前目标。也不能只比较数值：新目标m/v初始为0、step为0，保存值可能非零；恢复就是要替换这些值，检查的是结构与表示，而不是要求旧值等于新值。


### 继续沿微型树做纸面匹配

| 变化 | 代码在哪里拒绝／怎样处理？ |
|---|---|
| NPZ缺state_array_0002 | 按archive[key]取值失败；没有补零路径 |
| NPZ数组shape为(3,)，manifest却写[2] | 第一层元数据核对失败 |
| 文件与manifest都写(3,)，当前weight.m为(2,) | 第二层目标shape核对失败 |
| 文件step为uint32，目标step为uint64 | 即使都为shape ()，目标dtype核对仍失败 |
| 把step改成JSON Python int，目标为MLX数组 | 数组／非数组类别不同，失败 |
| 保存内容少了weight.v或多了一个目标没有的路径 | 路径集合不同，失败 |
| 保存step=7，目标step=0，双方都为() uint64 | 此项结构检查允许替换；7是否与权重／metadata进度一致要另外核对 |

本函数按目标current_flat的遍历顺序收集(path,saved_value)，再tree_unflatten。因此路径决定归位，不能按两个无标签数组列表的顺序盲目zip。

现有协议面向本课程生成并配对的文件，不是通用不可信manifest校验器。它不在此计算SHA-256、证明权重与state来自同一run，或保存任意空容器／自定义类；这些不能从“shape匹配”推断出来。


### 完整恢复函数：先重建候选，再赋回

按代码的空行阅读四段：①格式与文件读取；②从manifest建立path_to_value；③对照当前state逐叶核对；④tree_unflatten后才执行optimizer.state = rebuilt，最后mx.eval。

结构／类型检查在赋值前失败时，目标optimizer.state未被替换。最后的mx.eval发生在赋值之后，不提供失败回滚；不能把整个加载过程称为事务。array_file记录文件名，但本接口实际使用调用者传入的arrays_path，不自动搜索最新文件或替你核对配对路径。


In [ ]:
def restore_optimizer_state_from_files(optimizer, arrays_path, manifest_path):
    with Path(manifest_path).open("r", encoding="utf-8") as f:
        manifest = json.load(f)
    if manifest.get("format_version") != 2:
        raise ValueError("Unsupported optimizer-state format version.")
    if manifest.get("optimizer_class") != type(optimizer).__name__:
        raise ValueError("Optimizer class does not match saved state.")

    archive = mx.load(str(arrays_path))
    path_to_value = {}
    for item in manifest["arrays"]:
        value = archive[item["key"]]
        if list(value.shape) != item["shape"] or str(value.dtype) != item["dtype"]:
            raise ValueError(f"Saved array metadata mismatch at {item['path']}")
        path_to_value[item["path"]] = value
    for item in manifest["python_scalars"]:
        value = item["value"]
        expected_type = item["python_type"]
        if expected_type == "NoneType":
            if value is not None:
                raise ValueError(f"Scalar type mismatch at {item['path']}")
        elif type(value).__name__ != expected_type:
            raise ValueError(f"Scalar type mismatch at {item['path']}")
        path_to_value[item["path"]] = value

    # Validate against the target optimizer BEFORE mutating optimizer.state.
    current_flat = tree_flatten(optimizer.state)
    current_paths = [str(path) for path, _ in current_flat]
    if set(current_paths) != set(path_to_value):
        raise ValueError("Saved optimizer-state paths do not match the current optimizer.")
    rebuilt_items = []
    for path, current_value in current_flat:
        path_text = str(path)
        saved_value = path_to_value[path_text]
        current_is_array = isinstance(current_value, mx.array)
        saved_is_array = isinstance(saved_value, mx.array)
        if current_is_array != saved_is_array:
            raise ValueError(f"Array/non-array identity mismatch at {path_text}")
        if current_is_array:
            if tuple(current_value.shape) != tuple(saved_value.shape):
                raise ValueError(f"Shape mismatch at {path_text}: {current_value.shape} vs {saved_value.shape}")
            if current_value.dtype != saved_value.dtype:
                raise ValueError(f"Dtype mismatch at {path_text}: {current_value.dtype} vs {saved_value.dtype}")
        elif type(current_value) is not type(saved_value):
            raise ValueError(f"Non-array leaf type mismatch at {path_text}")
        rebuilt_items.append((path, saved_value))

    # Only assign after every path/type/shape/dtype check has passed.
    rebuilt = tree_unflatten(rebuilt_items)
    optimizer.state = rebuilt
    mx.eval(optimizer.state)


## 6. 零模板怎样接回真实恢复顺序？

[20.1](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/3255a2975b7e716d95c4f92d76f25e21256e7f66/notebooks/87_step20_1_save_model_weights.ipynb)的权重与本节Adam状态分开保存，[20.3](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/3255a2975b7e716d95c4f92d76f25e21256e7f66/notebooks/89_step20_3_save_configuration_metadata.ipynb)记录架构、完整词表、进度和artifact路径／指纹。下面只说明相邻接口的衔接，不运行完整续训：

| 顺序 | 未来应准备／完成什么？ |
|---:|---|
| 1 | 用同一run的架构、真实词表映射及文件记录确认配对；相同vocab_size不代表相同token-ID映射 |
| 2 | 构造匹配模型与Adam，optimizer.init(model.trainable_parameters())建立待校验的状态模板 |
| 3 | 严格加载该run的模型权重；与模板的参数结构和dtype必须一致 |
| 4 | 用本节函数从对应NPZ＋manifest恢复Adam状态，而不是仅保留零m/v |
| 5 | 核对恢复step与metadata的completed_updates一致；之后才可讨论构造续训所需求导／state引用 |

这与[20.4已存在的恢复接口](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/3255a2975b7e716d95c4f92d76f25e21256e7f66/notebooks/90_step20_4_resume_training.ipynb)保持一致；其restore_optimizer_state_from_files仍是同一实现。训练前若曾缓存 `[model.state, optimizer.state]`，须注意state赋值会换成新的字典，不能继续把旧optimizer状态容器当作已恢复状态；后续流程重新取得同一恢复对象的引用。

固定版本的state setter会把内部_initialized设False；下一次update需要时会init，而init保留已有非空m/v字典，不是无条件把恢复的moments清零。不要为“修复初始化标记”手动清空state或新建另一优化器覆盖恢复结果。这里仅解释源码语义，不执行下一步更新。

当前学习率数组可以被恢复值替换，但完整scheduler、betas、eps与bias_correction仍需匹配原run。没有完整RNG／sampler等信息，也不能把“可继续优化”说成“精确复现原轨迹”。


## 7. 本节检查点

本节新增知识是“路径＋数组表示＋目标校验”。原save/restore协议和文件名保持，纸面微型树仅帮助追踪四个叶子；没有创建它的MLX数组或写出示例产物。

自查（不附答案）：

1. shape为()的step为什么应保存为NPZ数组，而不是为了方便转成JSON整数？
2. 一个文件的shape/dtype都与manifest一致，为什么仍可能不适合目标optimizer？
3. 文件里的step与目标初始step值不相等，为什么不是本层结构检查的错误？还需对照什么？

以下附录只是已学、自包含前置；读懂序列化主线后再按需查看。


## 可选附录A · 既有MiniGPT定义

沿用[15.4完整MiniGPT](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/3255a2975b7e716d95c4f92d76f25e21256e7f66/notebooks/65_step15_4_implement_minigpt.ipynb)与20.1，不改结构、参数命名或dtype策略。MHA、MLP、Block和MiniGPT各一份，按类分开；它们不是本节的新序列化算法。模型构造前仍需同一真实词表的V，不能用纸面微型树代替实际模型。

MHA输入／输出为(B,T,C)，score为(B,H,T,T)，拆头见[13.3](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/3255a2975b7e716d95c4f92d76f25e21256e7f66/notebooks/54_step13_3_split_channels_into_heads.ipynb)。本节类定义不构造模型。


In [ ]:
import math
import mlx.nn as nn
import mlx.optimizers as optim

class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)


### 旧MLP与Block

[14.4](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/3255a2975b7e716d95c4f92d76f25e21256e7f66/notebooks/60_step14_4_feed_forward_mlp.ipynb)的带bias两层MLP与GELU、两条Pre-Norm RMSNorm残差路径保持不变。Block的输入／输出仍为(B,T,C)。


In [ ]:
class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))

class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


### 完整模型汇总

token IDs (B,T)经token/position embeddings与N个独立Block，输出raw logits (B,T,V)。本节保留原参数检查、final RMSNorm与untied LM head；不运行forward。


In [ ]:
class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        dims: int,
        num_heads: int,
        num_blocks: int,
        max_seq_len: int,
    ):
        super().__init__()

        if vocab_size <= 0:
            raise ValueError('vocab_size must be positive')
        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if num_blocks <= 0:
            raise ValueError('num_blocks must be positive')
        if max_seq_len <= 0:
            raise ValueError('max_seq_len must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        # Python metadata: these are not trainable mx.array parameters.
        self.vocab_size = vocab_size
        self.dims = dims
        self.max_seq_len = max_seq_len

        # Learned lookup tables.
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            dims=dims,
        )
        self.position_embedding = nn.Embedding(
            num_embeddings=max_seq_len,
            dims=dims,
        )

        # A list of distinct Module instances; MLX recursively tracks them.
        self.blocks = [
            TransformerBlock(
                dims=dims,
                num_heads=num_heads,
            )
            for _ in range(num_blocks)
        ]

        self.final_norm = nn.RMSNorm(dims=dims)
        self.lm_head = nn.Linear(
            input_dims=dims,
            output_dims=vocab_size,
            bias=False,
        )

    def __call__(self, token_ids: mx.array) -> mx.array:
        if len(token_ids.shape) != 2:
            raise ValueError('token_ids must have shape (B,T)')

        B, T = token_ids.shape

        if T < 1:
            raise ValueError('sequence length T must be at least 1')
        if T > self.max_seq_len:
            raise ValueError(
                f'sequence length {T} exceeds max_seq_len {self.max_seq_len}'
            )

        # Forward-local structural indices: not Module parameters.
        position_ids = mx.arange(T)  # (T,)

        token_features = self.token_embedding(token_ids)        # (B,T,C)
        position_features = self.position_embedding(position_ids)  # (T,C)
        x = token_features + position_features                  # (B,T,C)

        for block in self.blocks:
            x = block(x)  # invariant: (B,T,C)

        x = self.final_norm(x)  # (B,T,C)
        logits = self.lm_head(x)  # (B,T,V)

        expected_shape = (B, T, self.vocab_size)
        if logits.shape != expected_shape:
            raise RuntimeError(
                f'logits shape {logits.shape} != expected {expected_shape}'
            )

        return logits


## 可选附录B · 配置与零状态模板

MODEL_CONFIG的vocab_size仍为None，要求未来从真实token-ID映射取得，不读取语料来猜V。ADAM_CONFIG沿用原候选；create_model_and_optimizer只在被明确调用时构造新模型、创建Adam并init。它返回**新初始化对象**，不是加载函数。

用它创建目标模板有助于检查路径／shape／dtype；若直接保存新对象，保存的是零优化器历史，不能标记成训练后checkpoint。已有训练中的model/optimizer应保留匹配关系，不先调用此函数覆盖它们再声称保存了原状态。


In [ ]:
MODEL_CONFIG = {
    "vocab_size": None,  # Must come from the real token-ID mapping.
    "dims": 64,
    "num_heads": 4,
    "num_blocks": 2,
    "max_seq_len": 64,
}

def make_model_kwargs(config):
    if not isinstance(config["vocab_size"], int) or config["vocab_size"] < 1:
        raise ValueError("Set vocab_size from the real token-ID mapping.")
    if config["dims"] % config["num_heads"] != 0:
        raise ValueError("dims must be divisible by num_heads.")
    return dict(config)


In [ ]:
ADAM_CONFIG = {
    "learning_rate": 3e-4,
    "betas": [0.9, 0.999],
    "eps": 1e-8,
    "bias_correction": True,  # Explicit course choice; MLX v0.32.2 default is False.
}

def create_model_and_optimizer(vocab_size):
    config = dict(MODEL_CONFIG)
    config["vocab_size"] = vocab_size
    model = MiniGPT(**make_model_kwargs(config))
    optimizer = optim.Adam(**ADAM_CONFIG)
    optimizer.init(model.trainable_parameters())
    return model, optimizer


### 未来调用参考：先满足前置，再选择操作

下面仅为注释，三种操作不能盲目顺序执行：第一段是新模板，第二段要求已有明确来源的待保存optimizer，第三段要求权重已加载且目标模板／真实配对文件已准备。不是让你把新零状态保存后当成恢复成功；任何实际对象或文件缺失都不能伪造。

本节不提供真实路径、state或checkpoint，不自动读取磁盘，不调用以下任何函数。完整配对恢复按后续已存在材料处理，本轮不推进该节。


In [ ]:
# 新目标模板：real_vocab_size必须来自已确认的真实映射。
# model, optimizer = create_model_and_optimizer(real_vocab_size)

# 保存：仅对明确来源、与对应权重同一更新边界的optimizer使用。
# arrays_path, manifest_path = save_optimizer_state_to_new_directory(
#     optimizer, project_root / "results",
# )

# 恢复：target_optimizer已按匹配模型init；两路径来自配对记录。
# restore_optimizer_state_from_files(target_optimizer, arrays_path, manifest_path)


## 固定版本依据与材料状态

- [MLX v0.32.2 Optimizer/Adam](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/optimizers/optimizers.py)：零维uint64 step、learning_rate数组、m/v初始化、state setter与init行为。
- [v0.32.2 utils.py](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/utils.py)：路径展开与树重建约定。
- [v0.32.2 load.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/load.cpp)：NPZ中键与数组的保存／加载。
- [v0.32.2 array.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/array.cpp)：dtype字符串表示。

本次只整理20.2教学：纠正step在NPZ而非JSON标量的旧说法，修复数学公式显示，用微型树对照字段与两层校验，将旧模型移到可选附录。所有类、函数、配置与保存格式通过AST对照保持，不改变20.3/20.4兼容接口。

全部代码单元execution_count=null、outputs=[]。仅做JSON/schema、ID、语法、依赖与源码静态检查；没有运行Notebook、配置/测试代码、训练、验证、实验、采样、读取语料或真实state，未保存／加载checkpoint或操作用户工作副本。纸面示意不是运行结果，不声称掌握或运行通过。

20.2累计2/2（历史修正1次＋本次教学修订1次），其他历史计数保留。本轮停在20.2，等待独立验收，不推进其他节。
